# 🎵 Music Recommendation System — Collaborative Filtering on 30Music

This notebook walks through building a collaborative filtering recommender using the **30Music** dataset.  
We'll cover:
1. Parsing `.idomaar` files
2. Exploratory Data Analysis
3. Building a user-item interaction matrix
4. Collaborative Filtering with multiple approaches (user-based, item-based, matrix factorization)
5. Evaluation (Precision@K, Recall@K, NDCG)


## EDA:
Entities:

Persons -> artists

tracks-> song 5.6M

users-> age, gender, country, playcount (heavy listeners behave differnetly from normal ones)

Albums

playlists -> could have multiple users linked

tags -> genres, moods, artist names 

Relations:

events -> play event of each user,  and how long playtime is, playratio

session -> listening session of each user, playstart, playtime, playration, what type of songs the user listened to together

love -> explict songs which the user loved, high signal to find which songs user likes

---

## 0. Setup & Imports

In [8]:
# Install dependencies (uncomment if needed)
# !pip install pandas numpy scipy scikit-learn implicit matplotlib seaborn tqdm
import json
import os
import warnings
from collections import defaultdict

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy import sparse
from sklearn.model_selection import train_test_split
from sklearn.metrics.pairwise import cosine_similarity
from tqdm import tqdm

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

print('All imports successful!')

All imports successful!


## 1. Parsing the 30Music `.idomaar` Format

The `.idomaar` format stores one record per line with **tab-separated fields**:  
`timestamp \t entity_id \t type \t json_properties \t linked_entities`

We need parsers for:
- **entities/** → `users`, `tracks`, `albums`, `tags`, `persons`, `playlist`
- **relations/** → `events` (play events), `sessions`, `love` (explicit likes)

In [9]:
# ==============================
# SET YOUR DATASET PATH HERE
# ==============================
DATASET_ROOT = '/Users/yeshavyas/Documents/mlops_project/ThirtyMusic/'  # <-- change this to your path

ENTITIES_DIR = os.path.join(DATASET_ROOT, 'entities')
RELATIONS_DIR = os.path.join(DATASET_ROOT, 'relations')

# Verify the files exist
for d, label in [(ENTITIES_DIR, 'entities'), (RELATIONS_DIR, 'relations')]:
    if os.path.exists(d):
        print(f'{label}/: {os.listdir(d)}')
    else:
        print(f'⚠️  Directory not found: {d} — update DATASET_ROOT above')

entities/: ['tracks.idomaar', 'tags.idomaar', 'playlist.idomaar', 'persons.idomaar', 'albums.idomaar', 'users.idomaar']
relations/: ['love.idomaar', 'sessions.idomaar', 'events.idomaar']


In [10]:
def parse_idomaar(filepath, max_rows=None):
    """
    Parse a .idomaar file into a list of dicts.

    Each line: timestamp \t id \t type \t properties_json \t linked_entities_json
    Some files may have fewer fields; we handle that gracefully.
    """
    records = []
    with open(filepath, 'r', encoding='utf-8') as f:
        for i, line in enumerate(f):
            if max_rows and i >= max_rows:
                break
            line = line.strip()
            if not line:
                continue
            parts = line.split('\t')
            record = {
                'timestamp': parts[0] if len(parts) > 0 else None,
                'id': parts[1] if len(parts) > 1 else None,
                'type': parts[2] if len(parts) > 2 else None,
            }
            # Parse JSON properties
            if len(parts) > 3 and parts[3]:
                try:
                    record['properties'] = json.loads(parts[3])
                except json.JSONDecodeError:
                    record['properties'] = parts[3]
            else:
                record['properties'] = {}

            # Parse linked entities
            if len(parts) > 4 and parts[4]:
                try:
                    record['linked_entities'] = json.loads(parts[4])
                except json.JSONDecodeError:
                    record['linked_entities'] = parts[4]
            else:
                record['linked_entities'] = {}

            records.append(record)
    return records


def peek_file(filepath, n=3):
    """Quick look at first n records of an idomaar file."""
    records = parse_idomaar(filepath, max_rows=n)
    for r in records:
        print(json.dumps(r, indent=2, default=str)[:500])
        print('---')
    return records

print('Parser ready.')

Parser ready.


In [7]:
for label, directory in [("ENTITIES", ENTITIES_DIR), ("RELATIONS", RELATIONS_DIR)]:
    for filename in sorted(os.listdir(directory)):
        if filename.endswith(".idomaar"):
            filepath = os.path.join(directory, filename)
            print(f"=== {label}/{filename} ===")
            peek_file(filepath, n=2)
            print()


=== ENTITIES/albums.idomaar ===
{
  "timestamp": "album",
  "id": "0",
  "type": "-1",
  "properties": {
    "MBID": "81a775f4-4a0e-401f-b167-f8dd97453193",
    "title": "Mutazione: Italian Electronic & New Wave Underground 1980-1988"
  },
  "linked_entities": {}
}
---
{
  "timestamp": "album",
  "id": "1",
  "type": "-1",
  "properties": {
    "MBID": "189bd32e-65ec-46d6-b939-6e4988ac6872",
    "title": "Like What, Me Worry / Why Can't I Say"
  },
  "linked_entities": {}
}
---

=== ENTITIES/persons.idomaar ===
{
  "timestamp": "person",
  "id": "145148",
  "type": "-1",
  "properties": {
    "MBID": null,
    "name": "Everything+Is+Illuminated"
  },
  "linked_entities": {}
}
---
{
  "timestamp": "person",
  "id": "297899",
  "type": "-1",
  "properties": {
    "MBID": null,
    "name": "Robin+O%27Brien"
  },
  "linked_entities": {}
}
---

=== ENTITIES/playlist.idomaar ===
{
  "timestamp": "playlist",
  "id": "0",
  "type": "1216545588",
  "properties": {
    "ID": 2973549,
    "Title"

## 2. Load Core Data

For collaborative filtering, we primarily need **user–track interactions**.  
These come from:
- `relations/events.idomaar` → implicit feedback (play counts / listening events)
- `relations/love.idomaar` → explicit feedback (user "loved" a track)

We'll also load track metadata for interpretability.

In [ ]:
# ----------------------------
# Load EVENTS (play events)
# ----------------------------
print('Loading events (this may take a few minutes for large files)...')
events_raw = parse_idomaar(os.path.join(RELATIONS_DIR, 'events.idomaar'))
print(f'Loaded {len(events_raw):,} event records')

# Extract user_id, track_id, timestamp from events
events = []
for r in events_raw:
    props = r.get('properties', {})
    linked = r.get('linked_entities', {})

    # The exact field names depend on the dataset version.
    # Common patterns: subjects/objects in linked_entities, or userId/trackId in properties.
    # Adjust these based on what peek_file revealed above.
    user_id = None
    track_id = None

    # Pattern 1: linked_entities has subjects (users) and objects (tracks)
    if isinstance(linked, dict):
        subjects = linked.get('subjects', [])
        objects_ = linked.get('objects', [])
        if subjects:
            user_id = subjects[0].get('id') if isinstance(subjects[0], dict) else subjects[0]
        if objects_:
            track_id = objects_[0].get('id') if isinstance(objects_[0], dict) else objects_[0]

    # # Pattern 2: properties contain userId / trackId directly
    # if not user_id and isinstance(props, dict):
    #     user_id = props.get('userId') or props.get('user_id')
    #     track_id = props.get('trackId') or props.get('track_id') or props.get('itemId')

    # # Pattern 3: id field is user, and type encodes the track
    # if not user_id:
    #     user_id = r.get('id')

    if user_id and track_id:
        events.append({
            'user_id': str(user_id),
            'track_id': str(track_id),
            'timestamp': r.get('timestamp')
        })

events_df = pd.DataFrame(events)
print(f'Parsed {len(events_df):,} user-track play events')
events_df.head()

Loading events (this may take a few minutes for large files)...


In [ ]:
# ----------------------------
# Load LOVE (explicit likes)
# ----------------------------
print('Loading love data...')
love_raw = parse_idomaar(os.path.join(RELATIONS_DIR, 'love.idomaar'))
print(f'Loaded {len(love_raw):,} love records')

loves = []
for r in love_raw:
    linked = r.get('linked_entities', {})
    user_id = track_id = None
    if isinstance(linked, dict):
        subjects = linked.get('subjects', [])
        objects_ = linked.get('objects', [])
        if subjects:
            user_id = subjects[0].get('id') if isinstance(subjects[0], dict) else subjects[0]
        if objects_:
            track_id = objects_[0].get('id') if isinstance(objects_[0], dict) else objects_[0]
    if user_id and track_id:
        loves.append({'user_id': str(user_id), 'track_id': str(track_id), 'loved': 1})

love_df = pd.DataFrame(loves)
print(f'Parsed {len(love_df):,} love interactions')
love_df.head()

Loading love data...
Loaded 4,106,341 love records
Parsed 0 love interactions


""


In [ ]:
# ----------------------------
# Load TRACKS metadata
# ----------------------------
print('Loading tracks metadata...')
tracks_raw = parse_idomaar(os.path.join(ENTITIES_DIR, 'tracks.idomaar'))
print(f'Loaded {len(tracks_raw):,} tracks')

tracks = []
for r in tracks_raw:
    props = r.get('properties', {})
    name = props.get('name') or props.get('title') or 'Unknown'
    artist = props.get('artist') or props.get('artistname') or ''
    tracks.append({
        'track_id': str(r.get('id')),
        'name': name,
        'artist': artist
    })

tracks_df = pd.DataFrame(tracks)
print(f'Parsed {len(tracks_df):,} track records')
tracks_df.head()

## 3. Exploratory Data Analysis

In [ ]:
print('=== Dataset Summary ===')
n_users = events_df['user_id'].nunique()
n_tracks = events_df['track_id'].nunique()
n_events = len(events_df)
sparsity = 1 - (n_events / (n_users * n_tracks))

print(f'Users:        {n_users:>10,}')
print(f'Tracks:       {n_tracks:>10,}')
print(f'Interactions: {n_events:>10,}')
print(f'Sparsity:     {sparsity:>10.4%}')
print(f'Avg events/user: {n_events / n_users:.1f}')
print(f'Avg events/track: {n_events / n_tracks:.1f}')

In [ ]:
# Distribution of interactions per user and per track
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

user_counts = events_df['user_id'].value_counts()
axes[0].hist(user_counts.values, bins=50, color='steelblue', edgecolor='white', log=True)
axes[0].set_xlabel('Number of play events')
axes[0].set_ylabel('Number of users (log scale)')
axes[0].set_title('Distribution of Plays per User')
axes[0].axvline(user_counts.median(), color='red', ls='--', label=f'Median: {user_counts.median():.0f}')
axes[0].legend()

track_counts = events_df['track_id'].value_counts()
axes[1].hist(track_counts.values, bins=50, color='coral', edgecolor='white', log=True)
axes[1].set_xlabel('Number of play events')
axes[1].set_ylabel('Number of tracks (log scale)')
axes[1].set_title('Distribution of Plays per Track')
axes[1].axvline(track_counts.median(), color='red', ls='--', label=f'Median: {track_counts.median():.0f}')
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
# Top 15 most played tracks
top_tracks = (
    events_df['track_id'].value_counts()
    .head(15)
    .reset_index()
    .rename(columns={'index': 'track_id', 'track_id': 'play_count'})
)
# Merge with track names if available
if 'track_id' in top_tracks.columns and len(tracks_df) > 0:
    top_tracks = top_tracks.merge(tracks_df, on='track_id', how='left')
    top_tracks['label'] = top_tracks['artist'] + ' - ' + top_tracks['name']
else:
    top_tracks['label'] = top_tracks['track_id']

top_tracks

## 4. Build User-Item Interaction Matrix

We'll create an implicit feedback matrix where each entry is the **number of times** a user played a track. We also apply some standard preprocessing:
- Filter cold-start users (< 5 interactions)
- Filter cold-start items (< 3 interactions)
- Map IDs to contiguous integer indices

In [ ]:
# Aggregate play counts
play_counts = (
    events_df
    .groupby(['user_id', 'track_id'])
    .size()
    .reset_index(name='play_count')
)
print(f'Unique user-track pairs: {len(play_counts):,}')

# Filter: keep users with >= 5 interactions and tracks with >= 3
MIN_USER_INTERACTIONS = 5
MIN_TRACK_INTERACTIONS = 3

for iteration in range(3):  # iterative filtering
    user_activity = play_counts.groupby('user_id').size()
    active_users = user_activity[user_activity >= MIN_USER_INTERACTIONS].index
    play_counts = play_counts[play_counts['user_id'].isin(active_users)]

    track_popularity = play_counts.groupby('track_id').size()
    active_tracks = track_popularity[track_popularity >= MIN_TRACK_INTERACTIONS].index
    play_counts = play_counts[play_counts['track_id'].isin(active_tracks)]

n_users_f = play_counts['user_id'].nunique()
n_tracks_f = play_counts['track_id'].nunique()
print(f'After filtering: {n_users_f:,} users, {n_tracks_f:,} tracks, {len(play_counts):,} interactions')
print(f'Sparsity: {1 - len(play_counts) / (n_users_f * n_tracks_f):.4%}')

In [ ]:
# Create ID mappings
user_ids = sorted(play_counts['user_id'].unique())
track_ids = sorted(play_counts['track_id'].unique())

user2idx = {uid: i for i, uid in enumerate(user_ids)}
idx2user = {i: uid for uid, i in user2idx.items()}
track2idx = {tid: i for i, tid in enumerate(track_ids)}
idx2track = {i: tid for tid, i in track2idx.items()}

play_counts['user_idx'] = play_counts['user_id'].map(user2idx)
play_counts['track_idx'] = play_counts['track_id'].map(track2idx)

# Build sparse matrix
interaction_matrix = sparse.csr_matrix(
    (play_counts['play_count'].values,
     (play_counts['user_idx'].values, play_counts['track_idx'].values)),
    shape=(len(user_ids), len(track_ids))
)

print(f'Interaction matrix shape: {interaction_matrix.shape}')
print(f'Non-zero entries: {interaction_matrix.nnz:,}')

## 5. Train/Test Split

For each user, we hold out **20%** of their interactions as test data (randomly sampled).  
This simulates: "given a user's past listening, can we predict what they'll listen to next?"

In [ ]:
def train_test_split_by_user(df, test_ratio=0.2, random_state=42):
    """Hold out a fraction of each user's interactions for testing."""
    np.random.seed(random_state)
    train_list, test_list = [], []

    for user_id, group in df.groupby('user_id'):
        n_test = max(1, int(len(group) * test_ratio))
        test_indices = np.random.choice(group.index, size=n_test, replace=False)
        train_list.append(group.drop(test_indices))
        test_list.append(group.loc[test_indices])

    return pd.concat(train_list), pd.concat(test_list)

train_df, test_df = train_test_split_by_user(play_counts)
print(f'Train: {len(train_df):,} interactions')
print(f'Test:  {len(test_df):,} interactions')

# Build train matrix
train_matrix = sparse.csr_matrix(
    (train_df['play_count'].values,
     (train_df['user_idx'].values, train_df['track_idx'].values)),
    shape=interaction_matrix.shape
)

# Test: ground truth sets per user
test_ground_truth = test_df.groupby('user_idx')['track_idx'].apply(set).to_dict()
print(f'Users with test data: {len(test_ground_truth):,}')

## 6. Evaluation Metrics

Standard top-K recommendation metrics.

In [ ]:
def precision_at_k(recommended, relevant, k):
    """Fraction of top-k recommendations that are relevant."""
    rec_k = set(recommended[:k])
    return len(rec_k & relevant) / k if k > 0 else 0.0

def recall_at_k(recommended, relevant, k):
    """Fraction of relevant items found in top-k."""
    rec_k = set(recommended[:k])
    return len(rec_k & relevant) / len(relevant) if relevant else 0.0

def ndcg_at_k(recommended, relevant, k):
    """Normalized Discounted Cumulative Gain at K."""
    dcg = sum(1.0 / np.log2(i + 2) for i, item in enumerate(recommended[:k]) if item in relevant)
    idcg = sum(1.0 / np.log2(i + 2) for i in range(min(len(relevant), k)))
    return dcg / idcg if idcg > 0 else 0.0

def evaluate_model(recommend_fn, test_truth, train_mat, k_values=[5, 10, 20], n_eval_users=500):
    """
    Evaluate a recommendation function.
    recommend_fn(user_idx, train_matrix, n) -> list of track_idx recommendations
    """
    results = {k: {'precision': [], 'recall': [], 'ndcg': []} for k in k_values}
    eval_users = list(test_truth.keys())[:n_eval_users]

    for user_idx in tqdm(eval_users, desc='Evaluating'):
        relevant = test_truth[user_idx]
        max_k = max(k_values)
        recommended = recommend_fn(user_idx, train_mat, max_k)

        for k in k_values:
            results[k]['precision'].append(precision_at_k(recommended, relevant, k))
            results[k]['recall'].append(recall_at_k(recommended, relevant, k))
            results[k]['ndcg'].append(ndcg_at_k(recommended, relevant, k))

    # Average
    summary = {}
    for k in k_values:
        summary[f'P@{k}'] = np.mean(results[k]['precision'])
        summary[f'R@{k}'] = np.mean(results[k]['recall'])
        summary[f'NDCG@{k}'] = np.mean(results[k]['ndcg'])

    return pd.DataFrame([summary])

print('Evaluation functions ready.')

## 7. Baseline — Popularity Recommender

Always recommend the globally most popular tracks (that the user hasn't heard).

In [ ]:
# Precompute global popularity from train data
track_popularity_train = np.array(train_matrix.sum(axis=0)).flatten()
popular_tracks = np.argsort(-track_popularity_train)  # descending

def recommend_popularity(user_idx, train_mat, n):
    """Recommend most popular tracks the user hasn't interacted with."""
    user_seen = set(train_mat[user_idx].nonzero()[1])
    recs = [t for t in popular_tracks if t not in user_seen][:n]
    return recs

print('--- Popularity Baseline ---')
pop_results = evaluate_model(recommend_popularity, test_ground_truth, train_matrix)
pop_results.index = ['Popularity']
pop_results

## 8. User-Based Collaborative Filtering

Find similar users via cosine similarity, then recommend tracks from their listening history.

In [ ]:
# Binarize for similarity computation (implicit feedback)
train_binary = (train_matrix > 0).astype(np.float32)

# Precompute user similarity (for small datasets; for large ones, use approximate methods)
# If dataset is very large, sample a subset or use LSH
print('Computing user-user cosine similarity...')
if train_binary.shape[0] <= 20000:
    user_sim = cosine_similarity(train_binary, dense_output=False)
    print(f'User similarity matrix: {user_sim.shape}')
else:
    print('Dataset too large for dense user similarity — using batched approach')
    user_sim = None  # We'll handle this in the recommend function

In [ ]:
def recommend_user_cf(user_idx, train_mat, n, k_neighbors=50):
    """
    User-based CF: find k most similar users, aggregate their tracks.
    """
    if user_sim is not None:
        sim_scores = np.array(user_sim[user_idx].todense()).flatten()
    else:
        # Compute on-the-fly for one user
        sim_scores = cosine_similarity(train_binary[user_idx], train_binary).flatten()

    sim_scores[user_idx] = 0  # exclude self
    top_neighbors = np.argsort(-sim_scores)[:k_neighbors]
    neighbor_weights = sim_scores[top_neighbors]

    # Weighted sum of neighbor interactions
    neighbor_matrix = train_mat[top_neighbors]
    scores = np.array(neighbor_weights @ neighbor_matrix).flatten()

    # Zero out already-seen tracks
    user_seen = set(train_mat[user_idx].nonzero()[1])
    for idx in user_seen:
        scores[idx] = 0

    top_items = np.argsort(-scores)[:n]
    return top_items.tolist()

print('--- User-Based CF ---')
ucf_results = evaluate_model(recommend_user_cf, test_ground_truth, train_matrix)
ucf_results.index = ['User-CF']
ucf_results

## 9. Item-Based Collaborative Filtering

Find similar tracks and recommend based on what a user already likes.

In [ ]:
# Item-item similarity
print('Computing item-item cosine similarity...')
if train_binary.shape[1] <= 30000:
    item_sim = cosine_similarity(train_binary.T, dense_output=False)
    print(f'Item similarity matrix: {item_sim.shape}')
else:
    print('Dataset too large for dense item similarity — using on-the-fly')
    item_sim = None

In [ ]:
def recommend_item_cf(user_idx, train_mat, n):
    """
    Item-based CF: for each track the user has played,
    find similar tracks and aggregate scores.
    """
    user_interactions = train_mat[user_idx]
    user_items = user_interactions.nonzero()[1]
    user_weights = np.array(user_interactions[0, user_items].todense()).flatten()

    if len(user_items) == 0:
        return recommend_popularity(user_idx, train_mat, n)

    # Aggregate item similarities weighted by play count
    if item_sim is not None:
        item_scores = np.array(user_weights @ item_sim[user_items].toarray()).flatten()
    else:
        item_vecs = train_binary[:, user_items].T
        sims = cosine_similarity(item_vecs, train_binary.T)
        item_scores = np.array(user_weights @ sims).flatten()

    # Zero out seen items
    for idx in user_items:
        item_scores[idx] = 0

    top_items = np.argsort(-item_scores)[:n]
    return top_items.tolist()

print('--- Item-Based CF ---')
icf_results = evaluate_model(recommend_item_cf, test_ground_truth, train_matrix)
icf_results.index = ['Item-CF']
icf_results

## 10. Matrix Factorization — ALS (Alternating Least Squares)

The `implicit` library provides a fast ALS implementation designed for implicit feedback.  
This is generally the strongest baseline for implicit CF.

In [ ]:
# Install implicit if needed
# !pip install implicit

try:
    from implicit.als import AlternatingLeastSquares
    from implicit.bpr import BayesianPersonalizedRanking
    HAS_IMPLICIT = True
    print('implicit library loaded successfully')
except ImportError:
    HAS_IMPLICIT = False
    print('⚠️  `implicit` not installed. Run: pip install implicit')

In [ ]:
if HAS_IMPLICIT:
    # ALS expects item-user matrix (transposed)
    # Apply confidence weighting: C = 1 + alpha * play_count
    alpha = 40
    train_confidence = (train_matrix * alpha).astype(np.float64)

    als_model = AlternatingLeastSquares(
        factors=128,
        regularization=0.01,
        iterations=20,
        random_state=42
    )

    print('Training ALS model...')
    als_model.fit(train_confidence.T.tocsr())  # expects item-user
    print('ALS training complete!')
else:
    print('Skipping ALS (implicit library not available)')

In [ ]:
if HAS_IMPLICIT:
    def recommend_als(user_idx, train_mat, n):
        """ALS-based recommendations."""
        ids, scores = als_model.recommend(
            user_idx,
            train_mat[user_idx],
            N=n,
            filter_already_liked_items=True
        )
        return ids.tolist()

    print('--- ALS Matrix Factorization ---')
    als_results = evaluate_model(recommend_als, test_ground_truth, train_matrix)
    als_results.index = ['ALS']
    display(als_results)

## 11. BPR (Bayesian Personalized Ranking)

An alternative matrix factorization approach optimized for ranking.

In [ ]:
if HAS_IMPLICIT:
    bpr_model = BayesianPersonalizedRanking(
        factors=128,
        regularization=0.01,
        iterations=100,
        random_state=42
    )

    print('Training BPR model...')
    bpr_model.fit(train_confidence.T.tocsr())
    print('BPR training complete!')

    def recommend_bpr(user_idx, train_mat, n):
        ids, scores = bpr_model.recommend(
            user_idx, train_mat[user_idx],
            N=n, filter_already_liked_items=True
        )
        return ids.tolist()

    print('--- BPR ---')
    bpr_results = evaluate_model(recommend_bpr, test_ground_truth, train_matrix)
    bpr_results.index = ['BPR']
    display(bpr_results)

## 12. Results Comparison

In [ ]:
# Combine all results
all_results = [pop_results, ucf_results, icf_results]
if HAS_IMPLICIT:
    all_results.extend([als_results, bpr_results])

comparison = pd.concat(all_results)
print('\n=== Model Comparison ===')
display(comparison.round(4))

In [ ]:
# Visualization
metrics_to_plot = ['NDCG@10', 'P@10', 'R@10']
available_metrics = [m for m in metrics_to_plot if m in comparison.columns]

fig, axes = plt.subplots(1, len(available_metrics), figsize=(5 * len(available_metrics), 5))
if len(available_metrics) == 1:
    axes = [axes]

colors = ['#636EFA', '#EF553B', '#00CC96', '#AB63FA', '#FFA15A']

for ax, metric in zip(axes, available_metrics):
    values = comparison[metric]
    bars = ax.barh(values.index, values.values, color=colors[:len(values)])
    ax.set_xlabel(metric)
    ax.set_title(metric)
    for bar, val in zip(bars, values.values):
        ax.text(bar.get_width() + 0.001, bar.get_y() + bar.get_height()/2,
                f'{val:.4f}', va='center', fontsize=10)

plt.suptitle('Collaborative Filtering Model Comparison on 30Music', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

## 13. Example Recommendations for a User

In [ ]:
def show_user_recommendations(user_idx, recommend_fn, model_name, n=10):
    """Display what a user has listened to and what we recommend."""
    # What they listened to (train)
    listened = train_matrix[user_idx].nonzero()[1]
    listened_counts = np.array(train_matrix[user_idx, listened].todense()).flatten()
    top_listened = listened[np.argsort(-listened_counts)[:10]]

    print(f'=== User {idx2user[user_idx]} ===')
    print(f'\nTop listened tracks ({len(listened)} total):')
    for tidx in top_listened:
        tid = idx2track[tidx]
        match = tracks_df[tracks_df['track_id'] == tid]
        if len(match) > 0:
            row = match.iloc[0]
            print(f'  {row["artist"]} - {row["name"]} (played {train_matrix[user_idx, tidx]:.0f}x)')
        else:
            print(f'  Track {tid} (played {train_matrix[user_idx, tidx]:.0f}x)')

    recs = recommend_fn(user_idx, train_matrix, n)
    print(f'\n{model_name} Recommendations:')
    for i, tidx in enumerate(recs, 1):
        tid = idx2track[tidx]
        match = tracks_df[tracks_df['track_id'] == tid]
        if len(match) > 0:
            row = match.iloc[0]
            print(f'  {i}. {row["artist"]} - {row["name"]}')
        else:
            print(f'  {i}. Track {tid}')

# Show for a sample user
sample_user = list(test_ground_truth.keys())[0]

if HAS_IMPLICIT:
    show_user_recommendations(sample_user, recommend_als, 'ALS')
else:
    show_user_recommendations(sample_user, recommend_user_cf, 'User-CF')

## 14. Next Steps

Ideas to extend this:

**Incorporate more data:**
- Use `love.idomaar` as a stronger signal (explicit feedback)
- Use `sessions.idomaar` for session-based sequential recommendations
- Use `tags.idomaar` for content features in a hybrid model

**Advanced models:**
- LightFM (hybrid CF + content features)
- Neural Collaborative Filtering (NCF)
- GRU4Rec for session-based recommendations
- VAE-CF (Variational Autoencoder for CF)

**Better evaluation:**
- Time-based split (train on past, test on future)
- Coverage and diversity metrics
- Novelty analysis (are we just recommending popular items?)
- A/B test simulation